In [5]:
!pip install langchain langchain-community langchain-groq chromadb sentence-transformers pypdf streamlit pyngrok -q

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [7]:
os.environ["GROQ_API_KEY"] = "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"

In [8]:
import urllib.request
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

papers = {
    "attention": "https://arxiv.org/pdf/1706.03762.pdf",
    "bert": "https://arxiv.org/pdf/1810.04805.pdf",
    "gpt3": "https://arxiv.org/pdf/2005.14165.pdf",
    "rag": "https://arxiv.org/pdf/2005.11401.pdf",
    "sbert": "https://arxiv.org/pdf/1908.10084.pdf",
    "lora": "https://arxiv.org/pdf/2106.09685.pdf",
    "llama2": "https://arxiv.org/pdf/2307.09288.pdf",
}

os.makedirs("papers", exist_ok=True)

for name, url in papers.items():
    path = f"papers/{name}.pdf"
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {name}")

print("\nAll papers downloaded!")

Downloaded: attention
Downloaded: bert
Downloaded: gpt3
Downloaded: rag
Downloaded: sbert
Downloaded: lora
Downloaded: llama2

All papers downloaded!


In [9]:
from pathlib import Path

all_docs = []

for name, _ in papers.items():
    path = f"papers/{name}.pdf"
    loader = PyPDFLoader(path)
    docs = loader.load()

    for doc in docs:
        doc.metadata["paper_name"] = name

    all_docs.extend(docs)
    print(f"Loaded: {name} ({len(docs)} pages)")

print(f"\nTotal pages loaded: {len(all_docs)}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")

Loaded: attention (15 pages)
Loaded: bert (16 pages)
Loaded: gpt3 (75 pages)
Loaded: rag (19 pages)
Loaded: sbert (11 pages)
Loaded: lora (26 pages)
Loaded: llama2 (77 pages)

Total pages loaded: 239
Total chunks created: 1057


In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

print(f"Vector DB ready! Total vectors: {vectordb._collection.count()}")

/var/folders/mr/4sz0v7dx1bj2kftxmrg_c1100000gn/T/ipykernel_9698/4229717610.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6851.26it/s]


Vector DB ready! Total vectors: 5285


In [11]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2
)

retriever = vectordb.as_retriever(search_kwargs={"k": 4})

prompt_template = """You are an expert AI research assistant.
Use the following context from research papers to answer the question.
Always mention which paper the answer is from.
If the answer is not in the context, say "I don't have enough information from the provided papers."

Context: {context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

def ask_rag(question):
    docs = vectordb.similarity_search(question, k=4)
    context = "\n\n".join([f"[{d.metadata['paper_name']}]: {d.page_content}" for d in docs])
    final_prompt = prompt.format(context=context, question=question)
    response = llm.invoke(final_prompt)
    sources = list(set([d.metadata['paper_name'] for d in docs]))
    return response.content, sources

answer, sources = ask_rag("What is self-attention?")
print("Answer:", answer)
print("Sources:", sources)


Answer: I don't have enough information from the provided papers to determine what self-attention is. The context provided describes an attention function, but it does not mention self-attention specifically.
Sources: ['attention']


In [12]:
with open("app.py", "w") as f:
    f.write("""
import streamlit as st
import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

os.environ["GROQ_API_KEY"] = "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"

st.set_page_config(page_title="Research Paper QA", page_icon="📚")
st.title(" 📚Research Paper Assistant")
st.caption("Ask questions from 7 NLP research papers")

@st.cache_resource
def load_chain():
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embedding_model)
    llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.2)
    return llm, vectordb

llm, vectordb = load_chain()

prompt_template = \"\"\"You are an expert AI research assistant.
Use the following context from research papers to answer the question.
Always mention which paper the answer is from.
If the answer is not in the context, say "I do not have enough information from the provided papers."

Context: {context}

Question: {question}

Answer:\"\"\"

prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if query := st.chat_input("Ask about the research papers..."):
    st.session_state.messages.append({"role": "user", "content": query})
    with st.chat_message("user"):
        st.markdown(query)

    with st.chat_message("assistant"):
        with st.spinner("Searching papers..."):
            docs = vectordb.similarity_search(query, k=4)
            context = "\\n\\n".join([f"[{d.metadata['paper_name']}]: {d.page_content}" for d in docs])
            final_prompt = prompt.format(context=context, question=query)
            response = llm.invoke(final_prompt)
            sources = list(set([d.metadata['paper_name'] for d in docs]))
            answer = response.content
            full_response = f"{answer}\\n\\n**Sources:** {', '.join(sources)}"
            st.markdown(full_response)

    st.session_state.messages.append({"role": "assistant", "content": full_response})
""")

print("app.py updated!")

app.py updated!


In [13]:
import subprocess
import time
from pyngrok import ngrok

ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
time.sleep(3)

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(10)

ngrok.set_auth_token("3EocMJjNjerk41hkCXSG7nKWkAp_2r2yuXVxBjo3QNNQvUeqZ")
public_url = ngrok.connect(8501)
print(f"App is live at: {public_url}")

App is live at: NgrokTunnel: "https://earful-isotope-demanding.ngrok-free.dev" -> "http://localhost:8501"
